# Residential Time-of-Use Battery Agent

## In simple words
A household battery agent learns when to charge from the grid, remain idle, or discharge stored energy while electricity prices change during the day.

## What is the state?
The state tells the agent the current hour and how much energy is stored: `(hour, battery_level)`. The simulation has 12 hours.

## Actions
- `0 — IDLE`: keep the battery level the same.
- `1 — CHARGE`: add one unit if the battery is not full.
- `2 — DISCHARGE`: use one stored unit if the battery is not empty.

## Reward
The environment gives a reward after each action. Lower electricity cost generally means a better (less negative) reward. Invalid battery actions may receive a penalty.

**Hourly demand:** `[2, 2, 1, 1, 2, 3, 3, 3, 2, 2, 1, 1]`
**Hourly tariff:** `[2, 2, 2, 3, 4, 5, 6, 6, 5, 4, 3, 2]`
**Maximum battery level:** `4`
**Initial battery level:** `2`


## Your tasks
1. Complete `choose_action()` using epsilon-greedy selection: explore by choosing a random action sometimes; otherwise choose the action with the highest Q-value.
2. Complete `update_agent()` using the Q-learning formula shown in the code comments.
3. Run all cells from top to bottom and inspect the reward graph.
4. In your own words, explain what the Q-table learns.

## Goal
The agent should reduce the total grid-energy cost while keeping the battery within its limits.

In [ ]:
# Cell 2: Standard Libraries & Environment Parameters (LOCKED)
import numpy as np
import matplotlib.pyplot as plt
import random

np.random.seed(42)
random.seed(42)

HOURS = 12
MAX_BATTERY = 4
INITIAL_BATTERY = 2
DEMAND_PROFILE = [2, 2, 1, 1, 2, 3, 3, 3, 2, 2, 1, 1]
TARIFF = [2, 2, 2, 3, 4, 5, 6, 6, 5, 4, 3, 2]

NUM_ACTIONS = 3
NUM_EPISODES = 400
MAX_STEPS_PER_EPISODE = HOURS
ALPHA = 0.10
GAMMA = 0.90
EPSILON = 0.20


In [ ]:
# Cell 3: Environment Simulator (LOCKED)
class EnvironmentEngine:
    def __init__(self):
        self.state_count = HOURS * (MAX_BATTERY + 1)
        self.reset()

    def state_index(self, hour, battery):
        return hour * (MAX_BATTERY + 1) + battery

    def reset(self):
        self.hour = 0
        self.battery = INITIAL_BATTERY
        return self.state_index(self.hour, self.battery)

    def step(self, action):
        demand = DEMAND_PROFILE[self.hour]
        tariff = TARIFF[self.hour]
        grid_energy = demand
        reward = 0.0

        if action == 1:  # CHARGE
            if self.battery < MAX_BATTERY:
                self.battery += 1
                grid_energy += 1
            else:
                reward -= 2.0
        elif action == 2:  # DISCHARGE
            if self.battery > 0:
                self.battery -= 1
                grid_energy = max(0, grid_energy - 1)
            else:
                reward -= 2.0

        reward -= tariff * grid_energy

        self.hour += 1
        done = self.hour >= HOURS

        if done:
            pass
            next_state = self.state_index(HOURS - 1, self.battery)
        else:
            next_state = self.state_index(self.hour, self.battery)

        return next_state, reward, done


In [ ]:
# Cell 4: [TASK 1] Agent Decision Logic — Epsilon-Greedy Action Selection
class QLearningAgent:
    def __init__(self, num_states, num_actions, alpha=ALPHA, gamma=GAMMA, epsilon=EPSILON):
        self.num_states = num_states
        self.num_actions = num_actions
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.q_table = np.zeros((num_states, num_actions))

    def choose_action(self, state):
        # TASK 1: Implement epsilon-greedy action selection.
        # With probability epsilon, select a random action.
        # Otherwise, select the action with the highest Q-value.
        ### YOUR CODE HERE ###
        pass


In [ ]:
# Cell 5: [TASK 2] Q-Learning / Bellman Update
def update_agent(self, state, action, reward, next_state, done):
    # TASK 2: Implement the tabular Q-Learning update.
    # If done: target = reward
    # Otherwise: target = reward + gamma * max(Q(next_state, action))
    # Then apply: Q <- Q + alpha * (target - Q)
    ### YOUR CODE HERE ###
    pass

QLearningAgent.update_agent = update_agent


In [ ]:
# Cell 6: Simulation & Test Driver (LOCKED)
env = EnvironmentEngine()
agent = QLearningAgent(env.state_count, NUM_ACTIONS)

rewards_history = []
steps_history = []
battery_history = []

for ep in range(NUM_EPISODES):
    state = env.reset()
    total_reward = 0.0
    steps = 0
    done = False

    while not done and steps < MAX_STEPS_PER_EPISODE:
        action = agent.choose_action(state)
        next_state, reward, done = env.step(action)
        agent.update_agent(state, action, reward, next_state, done)
        state = next_state
        total_reward += reward
        steps += 1

    rewards_history.append(total_reward)
    steps_history.append(steps)
    battery_history.append(env.battery)

print(f"Simulation completed across {NUM_EPISODES} episodes.")
print(f"Average reward (last 20 episodes): {np.mean(rewards_history[-20:]):.2f}")
print(f"Average steps (last 20 episodes): {np.mean(steps_history[-20:]):.2f}")
print("Learned Q-table shape:", agent.q_table.shape)


In [ ]:
# Cell 7: Learning Visualization (LOCKED)
plt.figure(figsize=(10, 4))
plt.plot(rewards_history)
plt.title("Q-Learning Reward Trajectory")
plt.xlabel("Episode")
plt.ylabel("Cumulative Reward")
plt.grid(True)
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(battery_history)
plt.title("Final Battery Level per Episode")
plt.xlabel("Episode")
plt.ylabel("Battery Level")
plt.grid(True)
plt.show()
